# Context Managers

## Implementation of a context manager with magic methods

Create a class `StdoutRedirector` that can be used as a context manager. You can refer to the documentation of the [`ContextManager`](https://docs.python.org/fr/3/library/stdtypes.html#typecontextmanager) type. This class will store the value of `sys.stdout` upon entering the context and replace it with the argument provided at its creation. Upon exiting the context, the original standard output should be restored.


In [ ]:
# Your code here

### Solution

In [ ]:
import io
import sys


class StdoutRedirector:
  def __init__(self, new_stdout):
    self.new_stdout = new_stdout

  def __enter__(self):
    self.old_stdout = sys.stdout
    sys.stdout = self.new_stdout

  def __exit__(self, exc_type, exc_val, exc_tb):
    sys.stdout = self.old_stdout
    return False


with io.StringIO() as string_io:
  with StdoutRedirector(string_io):
    print("Hello World")
  print(f"StringIO value : {string_io.getvalue()}")

## Implementation of a context manager with a generator

Recreate the `StdoutRedirector` context but this time using the [`contextlib.contextmanager`](https://docs.python.org/3/library/contextlib.html#contextlib.contextmanager) method, using a generator.


In [ ]:
# Your code here

### Solution

In [ ]:
import contextlib
import io
import sys


@contextlib.contextmanager
def StdoutRedirector(new_stdout):
  try:
    old_stdout = sys.stdout
    sys.stdout = new_stdout
    yield
  finally:
    sys.stdout = old_stdout


with io.StringIO() as string_io:
  with StdoutRedirector(string_io):
    print("Hello World")
  print(f"StringIO value : {string_io.getvalue()}")

## Implementation of a context that manages closing a resource

Implement, with and without `contextlib.contextmanager`, a context manager that closes a resource with a `close` method provided as an argument regardless of what happens at the end of the context.


In [ ]:
# Your code here

### Solution

In [ ]:
import contextlib


@contextlib.contextmanager
def closing(resource):
  try:
    yield
  finally:
    resource.close()


class Closing:
  def __init__(self, resource):
    self.resource = resource

  def __enter__(self):
    pass

  def __exit__(self, exc_type, exc_val, exc_tb):
    self.resource.close()
    return False


class Resource:
  def close(self):
    print("Closed")

resource = Resource()
with Closing(resource):
  pass

## Reimplementation of the `contextlib.contextmanager` decorator (advanced)

Using a decorator that creates a class with the `__init__`, `__enter__`, and `__exit__` methods, reimplement the [`contextlib.contextmanager`](https://docs.python.org/3/library/contextlib.html#contextlib.contextmanager) decorator.


In [ ]:
# Your code here

### Solution

In [ ]:
from collections.abc import Callable, Iterator
from types import TracebackType
from typing import ContextManager
import io

def contextmanager[**P, T](generator: Callable[P, Iterator[T]]
                           ) -> Callable[P, ContextManager[T]]:
  class Wrapper:
    def __init__(self, *args: P.args, **kwargs: P.kwargs) -> None:
      self._generator = generator(*args, **kwargs)

    def __enter__(self) -> T:
      return next(self._generator)

    def __exit__[U: BaseException](self,
                                   exc_type: type[U] | None,
                                   exc_val: U | None,
                                   exc_tb: TracebackType | None) -> bool:
      try:
        if exc_type is None:
          next(self._generator)
        else:
          self._generator.throw(exc_val)
      except StopIteration:
        pass
      else:
        raise RuntimeError("Generator used as a context should yield only once")
      return True

  return Wrapper

@contextmanager
def suppress(exc_type):
  try:
    yield
  except exc_type:
    pass

with suppress(NameError):
  print(abcd)

with suppress(NameError):
  1 / 0